In [ ]:
# ============================================================
# Maritime Logistics Operations Analysis
# Ondatax DataDNA Challenge — March 2026
# ============================================================
# Script: 01_data_cleaning.py
# Purpose: Load raw 4-table star schema, perform data quality
#          checks, remove duplicates, fix data types, enrich
#          with calculated fields, merge into analysis-ready dataset
# Input:   data/raw/*.csv
# Output:  data/processed/q3_master_table.csv
# ============================================================

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime


In [3]:
fact=pd.read_csv("fact_cargo_movements.csv")
terminal=pd.read_csv("dim_terminal.csv")
time=pd.read_csv("dim_time.csv")
vessel=pd.read_csv("dim_vessel.csv")

In [4]:
print(f"Loaded: fact={fact.shape},vessel={vessel.shape},terminal={terminal.shape},time={time.shape}")

Loaded: fact=(15000, 9),vessel=(1000, 5),terminal=(50, 4),time=(1461, 6)


In [5]:
def data_quality_check(fact,terminal,time,vessel):
    report={
        'fact_total_record':len(fact),
        'fact_duplicates':fact.duplicated(subset=['movement_id']).sum(),
        'missing_values':fact.isnull().sum().to_dict(),
        'negative_values':(fact.select_dtypes(include=np.number)<0).sum().to_dict(),

        'oprhan_terminal_ids':len(set(fact['terminal_id'])-set(terminal['terminal_id'])),
        'orphan_vessel_ids':len(set(fact['vessel_id'])-set(vessel['vessel_id'])),
        'orphan_date_ids':len(set(fact['date_id'])-set(time['date_id'])),

        'vessel_build_year':vessel[vessel['build_year']<1990].shape[0],

        'move_duration_outliers':fact[
            np.abs(fact['move_duration']-fact['move_duration'].mean())>3*fact['move_duration'].std()
        ].shape[0],

        'move_duration_near_zero':fact[fact['move_duration']<0].shape[0],
    }
    return report

quality_report=data_quality_check(fact,terminal,time,vessel)
print("---Quality Report---")
for k,v in quality_report.items():
    print(f"{k}: {v}")

---Quality Report---
fact_total_record: 15000
fact_duplicates: 13999
missing_values: {'movement_id': 0, 'date_id': 0, 'vessel_key': 0, 'vessel_id': 0, 'vessel_category': 0, 'terminal_id': 0, 'container_count': 0, 'move_duration': 0, 'route_geometry': 0}
negative_values: {'movement_id': 0, 'vessel_id': 0, 'terminal_id': 0, 'container_count': 0, 'move_duration': 0}
oprhan_terminal_ids: 0
orphan_vessel_ids: 0
orphan_date_ids: 0
vessel_build_year: 721
move_duration_outliers: 0
move_duration_near_zero: 0


In [6]:
initial_count=len(fact)
fact=fact.drop_duplicates(subset=['movement_id'],keep='first')
print(f"Duplicated Rows:{initial_count-len(fact)}")

Duplicated Rows:13999


In [7]:
fact['date_id']=pd.to_datetime(fact['date_id'],errors='coerce')
time['date_id']=pd.to_datetime(time['date_id'],errors='coerce')

fact['container_count']=pd.to_numeric(fact['container_count'],errors='coerce')
fact['move_duration']=pd.to_numeric(fact['move_duration'],errors='coerce')
vessel['build_year']=pd.to_numeric(vessel['build_year'],errors='coerce')

fact['vessel_category']=fact['vessel_category'].str.strip().str.title()
terminal['regional_hub']=terminal['regional_hub'].str.strip().str.upper()
time['shift']=time['shift'].str.strip().str.title()
vessel['vessel_category']=vessel['vessel_category'].str.strip().str.title()

print('Data Types Fixed')

Data Types Fixed


In [8]:
# Check first, then act
if fact['move_duration'].isna().sum() > 0:
    fact['move_duration'] = fact['move_duration'].fillna(fact['move_duration'].median())
    print("Filled move_duration nulls with median")
else:
    print("move_duration: no nulls found, skipping fillna")

move_duration: no nulls found, skipping fillna


In [9]:
fact['date_parse_error']=fact['date_id'].isna()
print(f"Rows with unparseable dates:{fact['date_parse_error'].sum()}")

Rows with unparseable dates:0


In [10]:
initial_count=len(fact)
fact=fact[~fact['date_parse_error']]

fact=fact[fact['move_duration']>=1]

fact=fact[fact['container_count']>=0]

print(f"Invalid records removed: {initial_count-len(fact)}")

duration_mean=fact['move_duration'].mean()
duration_std=fact['move_duration'].std()

fact['duration_outlier']=np.abs(fact['move_duration']-duration_mean)>3*duration_std
print(f"Outlier duration flagged(not removed): {fact['duration_outlier'].sum()}")

vessel['build_year_flag']=vessel['build_year']<1990
print(f"Outlier build year flagged(not removed): {vessel['build_year_flag'].sum()}")

Invalid records removed: 1
Outlier duration flagged(not removed): 0
Outlier build year flagged(not removed): 721


In [12]:
current_year=datetime.now().year
vessel['vessel_age']=current_year-vessel['build_year']
vessel.loc[vessel['build_year_flag'],'vessel_age']=np.nan

In [13]:
fact['suez_period']=pd.cut(
    fact['date_id'],
    bins=[
        pd.Timestamp('2020-01-01'),
        pd.Timestamp('2021-03-01'),
        pd.Timestamp('2021-06-01'),
        pd.Timestamp('2025-01-01'),
    ],
    labels=['pre_suez','suez_disruption','post_suez'],right=False
)

fact['year_month']=fact['date_id'].dt.to_period('M')

In [14]:
df=fact.merge(
    vessel[['vessel_id','vessel_category','build_year','vessel_age','build_year_flag']],
    on='vessel_id',how='left'
).merge(
    terminal[['terminal_id','terminal_name','regional_hub']],
    on='terminal_id',how='left'
).merge(
    time[['date_id','fiscal_year','quarter','month','week','shift']],
    on='date_id',how='left'
)

print(f"Final merged shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Final merged shape: (1601, 24)
Columns: ['movement_id', 'date_id', 'vessel_key', 'vessel_id', 'vessel_category_x', 'terminal_id', 'container_count', 'move_duration', 'route_geometry', 'date_parse_error', 'duration_outlier', 'suez_period', 'year_month', 'vessel_category_y', 'build_year', 'vessel_age', 'build_year_flag', 'terminal_name', 'regional_hub', 'fiscal_year', 'quarter', 'month', 'week', 'shift']


In [16]:
df['data_quality_flag']=np.where(
    df['build_year_flag'].fillna(False) | df['duration_outlier'],
    'Review',
    'Clean'
)

print(f"Quality distribution:\n{df['data_quality_flag'].value_counts()}")
print(f"Suez period distribution:\n{df['suez_period'].value_counts()}")
print(f"Shift distribution:\n{df['shift'].value_counts()}")

Quality distribution:
data_quality_flag
Review    1128
Clean      473
Name: count, dtype: int64
Suez period distribution:
suez_period
post_suez          1469
suez_disruption      68
pre_suez             64
Name: count, dtype: int64
Shift distribution:
shift
Night    808
Day      793
Name: count, dtype: int64


In [17]:
df.to_csv('maritime_analysis_ready.csv',index=False)
print("\n Saved: maritime_analysis_ready.csv")
print(f"Final record count: {len(df)}")


 Saved: maritime_analysis_ready.csv
Final record count: 1601
